In [ ]:
import pandas as pd

In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from essential.gpu_utils import select_best_gpus

select_best_gpus(1)
import os
import scanpy as sc

import plotnine as gg
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm
from scvi.external import MRVI
from matplotlib.colors import to_hex
from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import fcluster

import seaborn as sns


def get_dendrogram(dists):
    ds = squareform(dists)
    Z = linkage(ds, method="complete")
    # Z = optimal_leaf_ordering(Z, ds)
    return Z


tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"


adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]
adata.obs["target_"] = adata.obs["target"].astype(str)
adata.obs["initial_clustering"] = "0"


# specific processing
adata.obs_names_make_unique()
adata.obs_names = ["c_{}".format(i) for i in range(adata.n_obs)]
adata.obs.index.name = "cell_name"

In [ ]:
adata_ = adata.copy()
adata_ = adata_[~adata_.obs["gene"].isna()].copy()
adata_.obs["gene"] = adata_.obs["gene"].astype(str)
adata_

In [ ]:
# pd.set_option("display.max_rows", None)
# adata.obs.groupby("spacer")["rt_bc"].value_counts()

In [ ]:
MRVI.setup_anndata(adata_, layer="reads", sample_key="gene", batch_key="rt_bc")
model = MRVI(adata_)
model.train()

In [ ]:
adata_sub = adata_.copy()
sc.pp.subsample(adata_sub, n_obs=1000)

In [ ]:
dists = model.get_local_sample_distances(
    adata_sub, keep_cell=False, groupby="initial_clustering", batch_size=32
)
d1 = dists.loc[{"initial_clustering_name": "0"}]["initial_clustering"]

Z = get_dendrogram(d1)

In [ ]:
control_keys = dists["sample_x"][dists["sample_x"].str.startswith("Control")]
d_to_control = d1.loc[{"sample_x": control_keys}].min(axis=0)
d_to_control

In [ ]:
sns.clustermap(
    d1.to_pandas(),
    row_linkage=Z,
    col_linkage=Z,
    xticklabels=False,
    yticklabels=False,
    vmax=0.5,
)

In [ ]:
tsne = TSNE(n_components=2, random_state=0, metric="precomputed", init="random", perplexity=100)
X_tsne = tsne.fit_transform(d1.values)

In [ ]:
labels_k = fcluster(Z, t=26, criterion="maxclust")

rep_df = (
    pd.DataFrame(X_tsne, index=d1["sample_x"].values, columns=["tsne1", "tsne2"])
    .reset_index()
    .assign(
        clustering=labels_k,
        d_to_control=d_to_control,
    )
    .rename(columns={"index": "gene"})
    .merge(
        model.sample_info,
        on="gene",
        how="left",
    )
)
rep_df

In [ ]:
# import plotly.express as px

# fig = px.scatter(
#     rep_df,
#     x="tsne1",
#     y="tsne2",
#     hover_name="target",
#     title="t-SNE Visualization",
#     width=1000,
#     height=800,
# )
# fig.show()

In [ ]:
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="tsne1", y="tsne2", color="factor(clustering)"), size=0.5)
    + gg.theme(legend_position="none")
)

In [ ]:
rep_df["d_to_control"].plot.hist(bins=100)

In [ ]:
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="tsne1", y="tsne2", color="d_to_control"), size=0.5)
    + gg.scale_color_continuous(limits=(0, 0.25))
)

In [ ]:
for cluster in range(1, len(rep_df["clustering"].unique()) + 1):
    print(cluster, rep_df.loc[lambda x: x.clustering == cluster]["d_to_control"].mean())
    print(", ".join(rep_df.loc[lambda x: x.clustering == cluster].gene.unique()))
    print()

In [ ]:
rep_df.loc[lambda x: x["gene"].str.startswith("Control")]["T4"].plot.hist(bins=100)

In [ ]:
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="tsne1", y="tsne2"), size=2)
    + gg.geom_text(
        rep_df.loc[lambda x: x.gene.str.startswith("rpo")],
        gg.aes(x="tsne1", y="tsne2", label="gene"),
        size=10,
        color="red",
    )
    + gg.geom_text(
        rep_df.loc[lambda x: x.gene.str.startswith("dna")],
        gg.aes(x="tsne1", y="tsne2", label="gene"),
        size=10,
        color="green",
    )
    + gg.geom_point(
        rep_df.loc[lambda x: x.gene.str.contains("Control")],
        gg.aes(x="tsne1", y="tsne2"),
        color="blue",
    )
)

In [ ]:
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="tsne1", y="tsne2", color="T4"), size=0.5)
    + gg.scale_color_continuous(limits=(-1, 1))
)

In [ ]:
rep_df.sort_values("target")

In [ ]:
mapper = {
    1: "Non-coding RNAs and Prophage elements",
    2: "Ribose Transport",
    3: "Co-translational Protein Targeting and Quality Control",
    4: "Central Carbon Metabolism (Glycolysis/TCA entry)",
    5: "Lipid and Cofactor Biosynthesis",
    6: "Macromolecular Synthesis Machinery",
    7: "TCA Cycle (partial)",
    8: "Cell Envelope (LPS) and Replication",
    9: "Cell Envelope Biogenesis and Division",
    10: "Membrane Lipid Biosynthesis",
    11: "Translation (tRNAs) and Mobile Elements",
    12: "Replication Elongation and Ribosomal RNA",
    13: "General Background / Unclustered",
    14: "Aerobic Respiration (Fragment)",
    15: "Stress Response and Motility Background",
    16: "ATP Synthase (F1 complex)",
    17: "Metabolic Regulation (Weak)",
    18: "Aerobic Respiration and Heme Synthesis",
    19: "Oxidative Phosphorylation / Electron Transport Chain",
    20: "Transcriptional Regulators",
    21: "Iron Transport",
    22: "tRNA Modification",
    23: "Translation Elongation",
    24: "Ribosome Assembly",
    25: "Peptidoglycan and Amino Acid Translation",
    26: "50S Ribosomal Subunit",
}

rep_df["annotated_cluster"] = rep_df["clustering"].map(mapper)

In [ ]:
import plotly.express as px

fig = px.scatter(
    rep_df,
    x="tsne1",
    y="tsne2",
    color="annotated_cluster",
    hover_name="target",
    title="t-SNE Visualization",
    width=1000,
    height=800,
)
fig.show()

In [ ]:
rep_df_detected = rep_df.query("annotated_cluster != 'General Background / Unclustered'").query(
    "annotated_cluster != 'Stress Response and Motility Background'"
)

fig = px.scatter(
    rep_df_detected,
    x="tsne1",
    y="tsne2",
    color="annotated_cluster",
    hover_name="target",
    title="t-SNE Visualization",
    width=1000,
    height=800,
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    rep_df,
    x="tsne1",
    y="tsne2",
    color="T4",
    hover_name="target",
    title="t-SNE Visualization",
    width=1000,
    height=800,
)
fig.show()

In [ ]:
all_reps = []
for perplexity in tqdm([10, 20, 30, 50, 100]):
    tsne = TSNE(
        n_components=2, random_state=0, metric="precomputed", init="random", perplexity=perplexity
    )
    X_tsne = tsne.fit_transform(d1.values)
    plot_df = pd.DataFrame(X_tsne, index=d1["sample_x"].values, columns=["tsne1", "tsne2"]).assign(
        perplexity=perplexity
    )
    all_reps.append(plot_df)
rep_df = pd.concat(all_reps)

In [ ]:
!pwd

In [ ]:
rep_df.to_csv("/workspace/experiments/01122025_ghosts/mrvi_discoveries.csv")

In [ ]:
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="tsne1", y="tsne2"), size=0.5)
    + gg.facet_wrap("perplexity", scales="free")
    + gg.theme(legend_position="none")
)

In [ ]:
adata_scvi = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
adata_scvi

In [ ]:
for cluster in rep_df_detected["annotated_cluster"].unique():
    print(cluster)
    print(
        ", ".join(
            rep_df_detected.loc[lambda x: x.annotated_cluster == cluster]
            .target.value_counts()
            .loc[lambda x: x >= 1]
            .index
        )
    )
    print()

In [ ]:
for cluster in adata_scvi.obs["annotated_leiden_case"].unique():
    print(cluster)
    print(
        ", ".join(
            adata_scvi.obs.loc[lambda x: x.annotated_leiden_case == cluster]
            .target.value_counts()
            .loc[lambda x: x >= 2]
            .index
        )
    )
    print()

In [ ]:
for cluster in adata_scvi.obs["annotated_leiden_case"].unique():
    print(cluster)
    print(
        ", ".join(
            adata_scvi.obs.loc[lambda x: x.annotated_leiden_case == cluster]
            .target.value_counts()
            .loc[lambda x: x >= 2]
            .index
        )
    )
    print()

In [ ]:
rep_df_detected